In [11]:
import pandas as pd

In [12]:
#Leitura da tabela, criação de coluna Despesa e conversões de datas
df_viagens = pd.read_csv('../data/raw/2023_Viagem.csv',encoding='Windows-1252', sep=';', decimal=',',thousands='.')
df_viagens["Despesas"] = df_viagens["Valor diárias"] + df_viagens["Valor passagens"] - df_viagens["Valor devolução"] + df_viagens["Valor outros gastos"]
df_viagens['Período - Data de início'] = pd.to_datetime(df_viagens['Período - Data de início'], format='%d/%m/%Y', errors='coerce')
df_viagens['Período - Data de fim'] = pd.to_datetime(df_viagens['Período - Data de fim'], format='%d/%m/%Y', errors='coerce')

In [13]:
pd.set_option("display.float_format", "{:.2f}".format)

## 1. Quais órgãos superiores mais gastam com viagens?

In [15]:
df_viagens.groupby('Nome do órgão superior')['Despesas'].sum().sort_values(ascending=False).head(10)

Nome do órgão superior
Sem informação                                   684826919.13
Ministério da Justiça e Segurança Pública        319666679.36
Ministério da Defesa                             302913410.72
Ministério da Educação                           270703578.49
Ministério do Meio Ambiente e Mudança do Clima   113392274.79
Ministério da Saúde                               86260605.24
Ministério das Relações Exteriores                75122342.20
Ministério da Previdência Social                  66849328.24
Ministério do Planejamento e Orçamento            52893221.89
Ministério dos Transportes                        49372783.23
Name: Despesas, dtype: float64

In [16]:
df_viagens[df_viagens['Nome do órgão superior'] == 'Sem informação']['Nome do órgão superior'].value_counts()

Nome do órgão superior
Sem informação    163659
Name: count, dtype: int64

Observação: "Sem informação" aparece no topo do ranking, representando 19,8% das viagens (163.659 de 826.989 registros) sem o órgão superior identificado. Esse volume expressivo de dados ausentes é, em si, um achado relevante sobre a qualidade da base do Portal da Transparência, e será destacado separadamente do ranking de órgãos identificados na apresentação final.

## 2. Quais os destinos mais frequentes?

In [17]:
df_viagens['Destinos'].head(10)

0               Loughborough/Reino Unido
1                      Rio de Janeiro/RJ
2    Nova York/Estados Unidos da América
3                             Vitória/ES
4                      Gotemburgo/Suécia
5      Informações protegidas por sigilo
6                      Gotemburgo/Suécia
7                      Gotemburgo/Suécia
8                      Gotemburgo/Suécia
9                               Natal/RN
Name: Destinos, dtype: str

In [18]:
df_viagens[df_viagens['Destinos'] == 'Informações protegidas por sigilo']['Destinos'].value_counts()

Destinos
Informações protegidas por sigilo    118222
Name: count, dtype: int64

In [19]:
df_viagens['Destinos'].value_counts()

Destinos
Informações protegidas por sigilo                   118222
Brasília/DF                                          64288
Rio de Janeiro/RJ                                    26228
São Paulo/SP                                         24621
São José dos Campos/SP                               12285
                                                     ...  
Rio de Janeiro/RJ, Chapecó/SC, Rio de Janeiro/RJ         1
Tabatinga/AM, Belém/PA                                   1
Boa Vista/RR, Alto Alegre/RS, Boa Vista/RR               1
Fernando de Noronha/PE, Recife/PE                        1
São José dos Pinhais/PR, Salvador/BA                     1
Name: count, Length: 37023, dtype: int64

In [20]:
df_destinos = df_viagens.copy()
df_destinos['Destinos'] = df_destinos['Destinos'].str.split(',')
df_destinos = df_destinos.explode('Destinos')
df_destinos['Destinos'] = df_destinos['Destinos'].str.strip()

In [22]:
df_destinos['Destinos'].value_counts().head(10)

Destinos
Informações protegidas por sigilo    118222
Brasília/DF                           76346
Rio de Janeiro/RJ                     31408
São Paulo/SP                          28696
São José dos Campos/SP                12978
Manaus/AM                             10588
Salvador/BA                           10369
Curitiba/PR                           10296
Belo Horizonte/MG                     10194
Recife/PE                              9977
Name: count, dtype: int64

Observação: "Informações protegidas por sigilo" lidera o ranking de destinos, representando 14,3% das viagens (118.222 de 826.989 registros) cujo destino não é divulgado por motivo de sigilo. Assim como o achado da Pergunta 1, esse volume será destacado separadamente do ranking de destinos identificados na apresentação final.

Nota metodológica: para este ranking, viagens com múltiplos destinos em uma mesma célula (ex: "Rio de Janeiro/RJ, Chapecó/SC") foram desmembradas em um destino por linha (str.split + explode), para que cada destino individual seja contado corretamente.

## 3: Quais os destinos mais caros?

In [28]:
df_viagens.sort_values(by='Despesas', ascending=False).head(10)[['Destinos', 'Despesas']]

,Destinos,Despesas
466711,"Jacarta/Indonésia, Phnom Penh/Camboja, Nova Yo...",195192.15
129058,"Xangai/China, São Paulo/SP",192676.02
129008,"Xangai/China, São Paulo/SP",192048.56
206998,Informações protegidas por sigilo,184787.42
1869,Lisboa/Portugal,172866.17
168228,"Xangai/China, Pequim/China",164821.02
504113,"Havana/Cuba, Nova York/Estados Unidos da América",164108.66
7339,"Brasília/DF, Brasília/DF, Rio de Janeiro/RJ, B...",163129.52
261765,Brasília/DF,157584.06
218975,Manama/Bareine,157255.53


Observação: das 10 viagens mais caras, 7 têm múltiplos destinos na mesma viagem, sugerindo
que viagens mais longas/complexas concentram os maiores gastos. O ranking mantém o valor
total da viagem e a lista completa de destinos como registrados originalmente, sem rateio
entre destinos.

## 4. Quais os 5 cargos que mais viajam?

In [32]:
df_cargo = df_viagens[df_viagens["Cargo"].notna()]

In [34]:
df_cargo.groupby('Cargo').size().sort_values(ascending=False).head(6)

Cargo
Informações protegidas por sigilo        118222
PROFESSOR DO MAGISTERIO SUPERIOR          54570
PROFESSOR ENS BASICO TECN TECNOLOGICO     37476
CONTRATADO LEI 8745/93 - NI               17787
AUDITOR FISCAL FEDERAL AGROPECUARIO       16582
ANALISTA AMBIENTAL                        13610
dtype: int64

Observação: "Informações protegidas por sigilo" também aparece na coluna Cargo, com 118.222 ocorrências — o mesmo volume da Pergunta 2, indicando que o sigilo é aplicado de forma consistente a todo o registro da viagem. Esse valor é destacado separadamente do ranking de cargos reais, seguindo o mesmo critério das perguntas anteriores. Linhas com Cargo nulo (~39% do total) foram excluídas desta análise específica, conforme decisão registrada na etapa de qualidade dos dados.

## 5. Qual a duração média das viagens?

In [35]:
df_viagens['Duração'] = (df_viagens['Período - Data de fim'] - df_viagens['Período - Data de início']).dt.days

In [41]:
df_viagens['Duração'].mean()

np.float64(5.893998590065889)

In [42]:
df_viagens['Duração'].median()

np.float64(3.0)

Duração média das viagens: 5,89 dias. Duração mediana: 3 dias.

A diferença expressiva entre média e mediana indica distribuição assimétrica: a maioria das viagens é curta (em torno de 3 dias), mas um número menor de viagens bem mais longas eleva a média. Recomenda-se apresentar as duas métricas juntas para não distorcer a leitura do dado típico.

## 6. Quais são os 10 cargos com mais despesas?

In [44]:
df_cargo.groupby('Cargo')['Despesas'].sum().sort_values(ascending=False).head(11)

Cargo
Informações protegidas por sigilo          367008551.67
PROFESSOR DO MAGISTERIO SUPERIOR           109857286.96
PROFESSOR ENS BASICO TECN TECNOLOGICO       52043200.47
TECNICO DO SEGURO SOCIAL                    45446081.28
ANALISTA AMBIENTAL                          34944744.39
ESP EM REGULACAO E VIGILANCIA SANITARIA     24253680.71
AUDITOR-FISCAL DA RECEITA FEDERAL BRASIL    24068986.38
AUDITOR FISCAL FEDERAL AGROPECUARIO         24040054.14
CONTRATADO LEI 8745/93 - NI                 17217779.17
TECNICO AMBIENTAL                           17152213.81
ANALISTA DO SEGURO SOCIAL                   15790180.45
Name: Despesas, dtype: float64

Observação: "Informações protegidas por sigilo" lidera com R$ 367 milhões, mais que o triplo do segundo colocado — mesmo padrão de peso do sigilo observado nas perguntas anteriores, excluído do ranking de cargos reais.

Professor do Magistério Superior e Professor Ens. Básico Técn. Tecnológico aparecem no topo tanto em número de viagens (Pergunta 4) quanto em despesas — são os cargos que mais viajam E mais gastam. Analista Ambiental, por outro lado, não estava no top 5 de viagens mas aparece no top 5 de despesas,  sugerindo viagens de custo médio mais alto para esse cargo.